# Phase 4: Router agent with LangGraph and MCP

This notebook routes each request to a scientist-style answer or a short play. The play writer can call the local MCP weather server when weather information is useful.

Run the cells in order. Keep open_weather.py in the same folder as this notebook.

## 1. Configure the local model and weather server

The chat model uses the local Ollama service. The MCP server is started with the same Python interpreter as the notebook. The file check gives a clear message if the weather server script is missing.

In [ ]:
import operator
import re
import sys
from pathlib import Path
from typing import Annotated, TypedDict

from IPython.display import display
import ipywidgets as widgets

from langchain_core.messages import (
    AnyMessage,
    HumanMessage,
    SystemMessage,
    ToolMessage,
)
from langchain_ollama import ChatOllama
from langgraph.graph import StateGraph, START, END

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from langchain_mcp_adapters.tools import load_mcp_tools

model = ChatOllama(
    model="qwen3.5:2b",
    temperature=0,
)


server_params = StdioServerParameters(
    command=sys.executable,
    args=[str(Path("open_weather.py").resolve())],
)

print("Local model and weather-server settings are ready.")

## 2. Define the router graph

The router classifies each user request. Requests for stories or plays go to play_writer; other questions go to scientist_writer. The state keeps messages from each node so the final response can be displayed.

In [ ]:
class AgentState(TypedDict):
    # Each node's new messages are appended to this list.
    messages: Annotated[list[AnyMessage], operator.add]


class RouterAgent:
    def __init__(self, model):
        self.model = model

        graph = StateGraph(AgentState)
        graph.add_node("router", self.router)
        graph.add_node("play_writer", self.play_writer)
        graph.add_node("scientist_writer", self.scientist_writer)

        graph.add_edge(START, "router")
        graph.add_conditional_edges(
            "router",
            self.router_decision,
            {
                "play": "play_writer",
                "scientist": "scientist_writer",
            },
        )
        graph.add_edge("play_writer", END)
        graph.add_edge("scientist_writer", END)

        self.graph = graph.compile()

    def router(self, state: AgentState):
        prompt = (
            "Classify the user's request. Reply with exactly one word: "
            "'play' if they ask for a story or play; otherwise reply "
            "'scientist'. Do not add an explanation."
        )

        # The user's question is the latest user message.
        user_message = next(
            message
            for message in reversed(state["messages"])
            if isinstance(message, HumanMessage)
        )

        answer = self.model.invoke([
            SystemMessage(content=prompt),
            user_message,
        ])
        return {"messages": [answer]}

    def router_decision(self, state: AgentState):
        answer = state["messages"][-1].content
        answer = re.sub(r"<think>.*?</think>", "", answer, flags=re.DOTALL)
        answer = answer.strip().lower()

        # Accept a little extra text from the small model, but choose a
        # valid graph path even if its classification is unexpected.
        if "play" in answer:
            return "play"
        return "scientist"

    async def play_writer(self, state: AgentState):
        user_message = next(
            message
            for message in reversed(state["messages"])
            if isinstance(message, HumanMessage)
        )

        prompt = (
            "Write a short, imaginative play in a Shakespeare-inspired style. "
            "If the request names a place, use the weather tool to get weather "
            "for that place and include it. Keep the play concise."
        )

        new_messages = []

        async with stdio_client(server_params) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await load_mcp_tools(session)
                model_with_tools = self.model.bind_tools(tools)

                messages = [
                    SystemMessage(content=prompt),
                    user_message,
                ]

                answer = await model_with_tools.ainvoke(messages)
                new_messages.append(answer)
                messages.append(answer)

                for tool_call in answer.tool_calls:
                    tool_result = await session.call_tool(
                        tool_call["name"],
                        arguments=tool_call["args"],
                    )

                    tool_text = "\n".join(
                        item.text
                        for item in tool_result.content
                        if hasattr(item, "text")
                    )

                    new_messages.append(
                        ToolMessage(
                            content=tool_text,
                            tool_call_id=tool_call["id"],
                        )
                    )
                    messages.append(new_messages[-1])

                if answer.tool_calls:
                    final_answer = await model_with_tools.ainvoke(messages)
                    new_messages.append(final_answer)

        return {"messages": new_messages}

    def scientist_writer(self, state: AgentState):
        user_message = next(
            message
            for message in reversed(state["messages"])
            if isinstance(message, HumanMessage)
        )

        answer = self.model.invoke([
            SystemMessage(
                content=(
                    "Answer in a clear, concise scientist's voice. "
                    "Explain the idea accurately for a general reader."
                )
            ),
            user_message,
        ])

        return {"messages": [answer]}


writer_agent = RouterAgent(model)
print("Router graph is ready.")

## 3. View the graph

This displays the LangGraph flow so you can see the router and its two response paths.

In [ ]:
display(
    {"text/vnd.mermaid": writer_agent.graph.get_graph().draw_mermaid()},
    raw=True,
)

## 4. Try the scientist route

This question should be sent to the scientist writer.

In [ ]:
science_result = await writer_agent.graph.ainvoke({
    "messages": [HumanMessage(content="Why is the sky blue?")]
})

print(science_result["messages"][-1].content)

## 5. Try the play route

This request should be sent to the play writer. The MCP server starts when this route runs. If it fails during startup, inspect the server error shown in the notebook output.

In [ ]:
play_result = await writer_agent.graph.ainvoke({
    "messages": [HumanMessage(content="Write a short play about the sea.")]
})

print(play_result["messages"][-1].content)

## What to notice

- The router chooses a path based on the request.
- The scientist node answers directly with the local model.
- The play node can access MCP weather tools.
- stdio_client sends MCP protocol messages over the server process's standard input/output; server diagnostics are directed to standard error.

## Exercises

- Ask for a different scientific explanation and compare it with the play route.
- Add another route, such as historian or engineer.
- Try a play request that mentions a city to explore the weather tool.